# Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sqlalchemy import create_engine

from underthesea import word_tokenize
import unicodedata
import re


from collections import defaultdict

# Connect to workbench

In [2]:
username = 'root'
password = '08042004'
host = 'localhost'
port = '3306'
database = 'law_db'

# Tạo engine kết nối đến MySQL
engine = create_engine(
    f"mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}?charset=utf8mb4"
)


# Lấy dữ liệu 

In [3]:
# Lấy dữ liệu
df_case = pd.read_sql("SELECT id, text, url, file FROM `case`", con = engine)

In [4]:
df_case.head()

,id,text,url,file
0,1,...,https://congbobanan.toaan.gov.vn/2ta737426t1cv...,https://congbobanan.toaan.gov.vn/5ta737426t1cv...
1,2,1 \n \nTÒA ÁN NHÂN DÂN \nCỘNG HÒA XÃ HỘI CHỦ N...,https://congbobanan.toaan.gov.vn/2ta162985t1cv...,https://congbobanan.toaan.gov.vn/5ta162985t1cv...
2,3,\n1 \nTÒA ÁN NHÂN DÂN QUẬN \nLONG BIÊN – TP H...,https://congbobanan.toaan.gov.vn/2ta162882t1cv...,https://congbobanan.toaan.gov.vn/5ta162882t1cv...
3,4,TÒA ÁN NHÂN DÂN \nCỘNG HÒA XÃ HỘI CHỦ N...,https://congbobanan.toaan.gov.vn/2ta158647t1cv...,https://congbobanan.toaan.gov.vn/5ta158647t1cv...
4,5,2 \nTÒA ÁN NHÂN DÂN \nHUYỆN QUAN SƠN \nTỈNH TH...,https://congbobanan.toaan.gov.vn/2ta141585t1cv...,https://congbobanan.toaan.gov.vn/5ta141585t1cv...


# I. Chuẩn bị dữ liệu

## Hàm tiền xử lý dữ liệu

In [5]:
def preprocess_text(text):
    # text k la str => rỗng
    if not isinstance(text, str):
        return ""
    # chữ thường
    text = text.lower()

    # chuyển từ viết tắt 
    abbrList = {
        'blhs': 'bộ luật hình sự',
        'tths': 'tố tụng hình sự',
        'thhs': 'thi hành hình sự',
        'blds': 'bộ luật dân sự',
        'ttds': 'tố tụng dân sự',
        'thds': 'thi hành dân sự'
    }

    for abbr, full_form in abbrList.items():
        text = text.replace(abbr, full_form)
    

    text = unicodedata.normalize("NFC", text) # chuẩn hóa mã hóa Unicode > đồng nhất
    # text = re.sub(r"[^a-zA-ZÀ-Ỹà-ỹ0-9\s]", " ", text) #giữ chữ, số , loại kí tự đbiet
    
    
    text = text.replace('\n', ' ')   # thay \n bằng khoảng trắng
    # khoảng trắng thừa = 1 space 
    while '  ' in text:
        text = text.replace('  ', ' ')
    text = text.strip() # xoa space đầu cuối 


    return text


In [6]:
# test 
text = "Blhs và tths quy định thế nào? Thhs, blds, ttds, thds liên quan. blhs.                          abc"
result = preprocess_text(text)
print(result)

bộ luật hình sự và tố tụng hình sự quy định thế nào? thi hành hình sự, bộ luật dân sự, tố tụng dân sự, thi hành dân sự liên quan. bộ luật hình sự. abc


In [7]:
import json

# Chọn 4 cột cần thiết
df_filtered = df_case[['id', 'url', 'file', 'text']].copy()

# Xử lý text
df_filtered['text'] = df_filtered['text'].apply(preprocess_text)

# Giới hạn id (ví dụ <= 1006)
max_id = 3000
min_id = 1800
df_filtered = df_filtered[(df_filtered['id'] >= min_id) & (df_filtered['id'] <= max_id)]


# Convert thành list of dicts
data = df_filtered.to_dict(orient="records")

# Xuất JSON

with open(f"cases_{min_id}_{max_id}.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

